# 10-2 機器學習進階：模型動物園、Ensemble、評估套餐與 SHAP

第一個 notebook（`10_ml_baseline`）誠實地讓你看到：**在 280 筆退伍軍人症資料上，Random Forest 幾乎贏不了 logistic regression**——真實疫情預測是謙卑的。那 ML 到底什麼時候才「贏得漂亮」？答案是：**當疾病風險是非線性、有交互作用、而且資料量夠大的時候。**

這個 notebook 換上一個更大的舞台，帶你走完一套完整的 ML 工作流：

- **模型動物園**：決策樹、隨機森林、XGBoost、LASSO——每個配一個臨床比喻
- **Ensemble**：bagging / boosting / **stacking（Super Learner）**
- **評估套餐**：ROC-AUC、PR-AUC、敏感度/特異度/PPV/NPV、校準（calibration）
- **SHAP**：把黑盒模型解釋給醫師聽
- **類別不平衡、過擬合、以及「ML 是工具，不是取代流病判斷」**

> 🏭 **換舞台：為什麼用「合成沙盒」？**
> 退伍軍人症只有 280 筆、訊號又弱，看不出 ML 的威力。所以這裡改用一份**合成的「區域級呼吸道疫情通報彙整資料」**（想像 CDC AI 辦公室把很多機構的通報彙整起來，n≈2500）。它是**教學沙盒**，刻意設計了「年齡的 U 型風險」和「免疫低下 × 暴露的交互作用」——正是 logistic 迴歸抓不到、而 ML 抓得到的那種訊號。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- 套件 ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
import xgboost as xgb
import shap

from epi_learning.viz import configure_chinese_font
configure_chinese_font()

## Step 1 — 打造教學沙盒（合成資料）

我們合成一份 2500 人的資料，`severe` = 是否演變成重症。刻意埋了兩個「ML 才抓得到」的訊號：

- **年齡的 U 型風險**：太年輕和太老都高風險（線性的 age 抓不到）
- **免疫低下 × 暴露的交互作用**：只有「免疫低下**且**高暴露」時風險才飆高

另外放了兩個**純雜訊**欄位（`noise_lab`、`noise_ward`），看模型會不會被騙。

In [ ]:
def make_cohort(seed=10, n=2500):
    rng = np.random.default_rng(seed)
    age            = rng.integers(20, 90, n)
    immunosuppressed = rng.binomial(1, 0.20, n)
    exposure       = rng.uniform(0, 1, n)          # 暴露劑量（氣霧/淋浴）
    vaccinated     = rng.binomial(1, 0.45, n)
    diabetes       = rng.binomial(1, 0.25, n)
    copd           = rng.binomial(1, 0.15, n)
    sex            = rng.binomial(1, 0.5, n)
    noise_lab      = rng.normal(0, 1, n)            # 純雜訊
    noise_ward     = rng.integers(0, 5, n)          # 純雜訊

    age_u = ((age - 55) / 20) ** 2                  # ← U 型：年輕與年老都高
    logit = (-3.4
             + 1.6 * age_u                          # 非線性
             + 2.6 * immunosuppressed * exposure    # ← 交互作用
             + 1.8 * exposure + 1.1 * diabetes + 1.0 * copd
             - 1.3 * vaccinated + 0.4 * sex)
    severe = rng.binomial(1, 1 / (1 + np.exp(-logit)))

    X = pd.DataFrame({"age": age, "immunosuppressed": immunosuppressed, "exposure": exposure.round(3),
                      "vaccinated": vaccinated, "diabetes": diabetes, "copd": copd, "sex": sex,
                      "noise_lab": noise_lab.round(2), "noise_ward": noise_ward})
    return X, pd.Series(severe, name="severe")

X, y = make_cohort()
print(f"樣本數 = {len(X)}，重症比例 = {y.mean():.1%}")
X.head()

## Step 2 — 三切分：train / validation / test（最重要的地基）

Ch07 教過「不能偷看未來」；ML 的版本是**三切分**：

| 資料集 | 比例 | 角色 | 白話 |
|---|---|---|---|
| **train** | 60% | 學公式（fit 模型） | 上課、寫作業 |
| **validation** | 20% | 調參數、選模型 | 模擬考（可以看答案檢討） |
| **test** | 20% | **只掀一次**的最終評估 | 期末考（看過就作廢） |

![train/val/test 三切分](../images/train_val_test_split.svg)

> 🚨 **資料洩漏（data leakage）是 ML 的頭號殺手**：只要「測試集的資訊」偷偷跑進訓練過程，模型就會考很高、上線卻慘敗。三大禁忌：① 先標準化/SMOTE 再切分（要在 fold **內部**做）；② 把「結果的一部分」當特徵（如用症狀預測感染）；③ 用到未來資訊。
>
> ⏳ **時間 / 空間資料要特別小心**：時間序列要用 `TimeSeriesSplit`（不能隨機打亂）；空間資料要用 spatial CV（別把相鄰區域拆散）——否則等於偷看。

In [ ]:
# 分層三切分：先切 70/30，再把 30 對半 → val 15% / test 15%（此處 test_size=0.5 讓 val≈test）
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=10, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=10, stratify=y_tmp)
print(f"train={len(X_train)}  validation={len(X_val)}  test={len(X_test)}")
print(f"三份的重症比例都接近：{y_train.mean():.2f} / {y_val.mean():.2f} / {y_test.mean():.2f}（stratify 的功勞）")

## Step 3 — 模型動物園（每個配一個臨床比喻）

- 🩺 **決策樹 Decision Tree = 急診檢傷分流圖**：一路問是非題（發燒？>65？免疫低下？）走到底分到高危/低危格。超好解釋，但問法太死、換一批病人整棵樹可能長歪（不穩、易過擬合）。
- 👥 **隨機森林 Random Forest = 多科會診投票（bagging）**：找幾百位醫師，每位只看**部分**病歷、**部分**檢查，各畫一棵樹、各投一票，多數決。沒人全知，但一群「略有差異、各自獨立」的醫師投票，比單一醫師穩。
- 📈 **XGBoost = 錯題本補習班（boosting）**：第一位老師教完，把「還是不會的錯題」挑出來丟給第二位專攻，第三位再補殘差……每一棒專心修正上一棒。強，但容易補過頭。
- 🧳 **LASSO（L1 邏輯斯迴歸）= 行李限重打包**：L1 懲罰像限重，逼不重要的變數係數**歸零**，只留幾個真正有用的因子 → 一張精簡、可寫進報告的清單。這就是流病最愛拿它當 baseline 的原因。

In [ ]:
zoo = {
    "LASSO":        LogisticRegression(penalty="l1", solver="liblinear", C=0.5,
                                       class_weight="balanced", max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=1, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=7, random_state=1,
                                            class_weight="balanced"),
    "XGBoost":       xgb.XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.07,
                                       eval_metric="logloss", random_state=1),
}

fitted, rows = {}, []
for name, model in zoo.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    p = model.predict_proba(X_test)[:, 1]
    rows.append({"模型": name, "AUC": roc_auc_score(y_test, p),
                 "PR-AUC": average_precision_score(y_test, p),
                 "Brier": brier_score_loss(y_test, p)})
zoo_scores = pd.DataFrame(rows).round(3)
print(zoo_scores.to_string(index=False))
print("\n→ LASSO（線性）AUC 只有 0.71；樹系模型 0.84+ ——這就是『非線性 + 交互作用』時 ML 贏的地方")

## Step 4 — Ensemble：三種「集思廣益」

- **Bagging**（隨機森林）：一群模型**平行**各看部分資料、投票平均 → 降低變異、更穩。
- **Boosting**（XGBoost）：模型**接力**，每一棒專攻上一棒的殘差 → 降低偏差、更準。
- 🎯 **Stacking = 指揮中心總指揮（Super Learner）**：樹、森林、XGBoost、LASSO 各給一個機率，**總指揮（meta 模型，通常 logistic）不自己看病人**，而是學「什麼情況該多聽哪位專家」，加權整合成最終判斷。這就是流病文獻的 **Super Learner**（van der Laan）。

In [ ]:
stack = StackingClassifier(
    estimators=[(k, v) for k, v in zoo.items()],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
)
stack.fit(X_train, y_train)
p_stack = stack.predict_proba(X_test)[:, 1]
print(f"Stacking (Super Learner) 測試集 AUC = {roc_auc_score(y_test, p_stack):.3f}")
print("→ Stacking 通常 ≥ 最佳單一模型，且較不會押錯寶（減少單一模型偏差）")

## Step 5 — 評估套餐：不要只看一個數字

不同任務看不同指標。一句話抓重點：**篩檢期**重「不能漏接」（sensitivity、NPV、PR-AUC）；**確認 / 資源分配期**重「別誤報、機率要準」（specificity、PPV、calibration）。

| 指標 | 何時看 | 流病白話 |
|---|---|---|
| **ROC-AUC** | 選模型、跨門檻 | 病人排在健康人前面的機率；不平衡時偏樂觀 |
| **PR-AUC** | **正例稀少**（重症/死亡） | 專注「抓到的陽性有多真」，比 AUC 誠實 |
| **Sensitivity** | 篩檢、漏接代價高 | 真病人裡抓到幾成 |
| **Specificity** | 誤報代價高 | 真沒病的裡正確放行幾成 |
| **PPV** | 臨床當下決策（受盛行率影響大） | 「模型說陽性，他真有病的機率」——醫師最在意 |
| **Calibration / Brier** | 要把機率當數字用（分床、風險溝通） | 說 70% 的那群，真的約 70% 發病嗎？**會排序 ≠ 機率準** |

In [ ]:
p_rf = fitted["Random Forest"].predict_proba(X_test)[:, 1]

# 混淆矩陣 @ 門檻 0.5 → 流病指標
tn, fp, fn, tp = confusion_matrix(y_test, (p_rf >= 0.5)).ravel()
sens, spec = tp / (tp + fn), tn / (tn + fp)
ppv, npv = tp / (tp + fp), tn / (tn + fn)
print(f"Random Forest @0.5：敏感度={sens:.2f} 特異度={spec:.2f} PPV={ppv:.2f} NPV={npv:.2f} F1={f1_score(y_test,(p_rf>=0.5)):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# ROC
fpr, tpr, _ = roc_curve(y_test, p_rf)
axes[0].plot(fpr, tpr, color="#D97757"); axes[0].plot([0,1],[0,1],"--",color="#6B6B6B")
axes[0].set_title(f"ROC 曲線 (AUC={roc_auc_score(y_test,p_rf):.2f})"); axes[0].set_xlabel("1 - 特異度"); axes[0].set_ylabel("敏感度")
# PR
prec, rec, _ = precision_recall_curve(y_test, p_rf)
axes[1].plot(rec, prec, color="#6A9BCC"); axes[1].axhline(y_test.mean(), ls="--", color="#6B6B6B")
axes[1].set_title(f"PR 曲線 (PR-AUC={average_precision_score(y_test,p_rf):.2f})"); axes[1].set_xlabel("Recall/敏感度"); axes[1].set_ylabel("Precision/PPV")
# Calibration
frac_pos, mean_pred = calibration_curve(y_test, p_rf, n_bins=5)
axes[2].plot(mean_pred, frac_pos, "o-", color="#788C5D"); axes[2].plot([0,1],[0,1],"--",color="#6B6B6B")
axes[2].set_title("校準圖 (預測機率 vs 實際發生率)"); axes[2].set_xlabel("預測機率"); axes[2].set_ylabel("實際比例")
plt.tight_layout(); plt.show()
print("→ 校準圖越貼近對角線，代表『說 70% 就真的約 70%』——想拿機率去分床，一定要看它")

## Step 6 — 類別不平衡：準確率的陷阱

真實 outbreak 常常「重症/死亡是少數」。這時**準確率會騙人**。

In [ ]:
# 把重症下採樣到 ~8%，做出一份不平衡資料
rng = np.random.default_rng(3)
pos_idx = y[y == 1].index
keep = rng.choice(pos_idx, size=int(len(pos_idx) * 0.12), replace=False)
imb_idx = y[y == 0].index.union(pd.Index(keep))
y_imb = y.loc[imb_idx]
print(f"不平衡資料：重症只佔 {y_imb.mean():.1%}")
print(f"👉 只要『全部猜沒重症』，準確率就有 {1 - y_imb.mean():.1%} —— 高得嚇人，但完全沒用（漏接每一個病人）！")
print("→ 所以不平衡時：① 別看 accuracy，改看 PR-AUC / 敏感度；")
print("  ② 用 class_weight='balanced'（或只在 train fold 內做 SMOTE，否則洩漏）把少數類的權重調高")

## Step 7 — SHAP：把黑盒解釋給醫師聽

> 💰 **年終公平分紅的比喻**：SHAP 用賽局理論的 Shapley value，問「少了這個特徵，預測差多少？」把每個特徵**加入 vs 不加入**的邊際貢獻，對所有加入順序平均。所以它能對**單一病人**說：他被判高風險，是因為「免疫低下 +0.3、暴露 +0.2、年齡 80 +0.15」——正好是對臨床醫師解釋黑盒的語言。

In [ ]:
explainer = shap.TreeExplainer(fitted["XGBoost"])
shap_values = explainer.shap_values(X_test)

# 1) 特徵重要性（beeswarm）：整體哪些特徵最重要
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout(); plt.show()

In [ ]:
# 2) 依賴圖：SHAP 揭露我們埋的『年齡 U 型』——年輕與年老 SHAP 值都往上翹
shap.dependence_plot("age", shap_values, X_test, interaction_index=None, show=False)
plt.tight_layout(); plt.show()
print("→ SHAP 成功還原了我們刻意設計的『年齡 U 型風險』——這是 logistic 的線性 age 永遠畫不出來的")

## Step 8 — 過擬合：訓練考 100 分，上線卻不及格

模型太複雜、資料太少，就會把**訓練集的雜訊也背起來**——訓練分數超高、測試分數卻很差。

In [ ]:
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=1).fit(X_train, y_train)  # 不限深度 → 過擬合
auc_tr = roc_auc_score(y_train, deep_tree.predict_proba(X_train)[:, 1])
auc_te = roc_auc_score(y_test,  deep_tree.predict_proba(X_test)[:, 1])
print(f"無限深度決策樹：訓練 AUC = {auc_tr:.3f}，測試 AUC = {auc_te:.3f}")
print(f"→ 訓練近乎完美、測試崩壞 = 典型過擬合。它把 noise_lab/noise_ward 這種雜訊也背起來了")
print("  對策：限制深度 / 剪枝、regularization（LASSO 的 L1）、early stopping、更多資料、交叉驗證")

## 收尾：ML 是「工具」，不是取代流病判斷

你走完了一套完整的 ML 工作流。但請把這幾句話刻在心裡——它們是**流行病學家**用 ML 和工程師最大的不同：

1. **會排序 ≠ 機率準**：AUC 高不代表能拿去分床，要看 **calibration**。
2. **重要 ≠ 有因果**：SHAP 說 `exposure` 重要，不代表「改變 exposure 就能防病」；feature importance **不是介入標的**（因果推論是 Ch12 的事）。
3. **外部驗證**：在松柏護理之家訓練的模型，搬去別家可能崩掉（時間/空間 dataset shift）——一定要在**新資料**上再驗一次。
4. **公平性**：不同性別、年齡層的 subgroup AUC 一致嗎？模型會不會對某群人特別不準？
5. **資料怎麼來，決定模型學到什麼**：通報偏差、選樣偏差會被模型忠實地學起來並放大。

> 🧭 **一句話**：ML 幫你**從一堆變數裡找出模式、做出預測**；但「這個模式**為什麼**存在、該**怎麼介入**、能不能**推廣**」，永遠是流行病學家的判斷。**模型是望遠鏡，不是方向盤。**